# SparseGaze Event Comparison Viewer

Experiment-local notebook for SparseGaze missing-frame evaluation. It compares SparseGaze rollout variants on `eval_mask` frames and groups errors by GT scene-gaze event labels.

In [2]:
from pathlib import Path
import importlib
import sys

EXPERIMENT_DIR = Path.cwd()
if EXPERIMENT_DIR.name != "sparsegaze_evaluation":
    EXPERIMENT_DIR = Path("/home/liumu/Github_Projects/adt_dataset_sandbox/Experiments/sparsegaze_evaluation")
REPO_ROOT = EXPERIMENT_DIR.parents[1]
sys.path.insert(0, str(EXPERIMENT_DIR))
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

import event_evaluation as eval_exp

eval_exp = importlib.reload(eval_exp)

plt.style.use("default")
plt.rcParams["figure.dpi"] = 130
pd.set_option("display.max_rows", 100)

print("experiment:", EXPERIMENT_DIR)
print("reports:", eval_exp.DEFAULT_REPORTS_DIR)
print("SparseGaze:", eval_exp.DEFAULT_SPARSEGAZE_DIR)

experiment: /home/liumu/Github_Projects/adt_dataset_sandbox/Experiments/sparsegaze_evaluation
reports: /mnt/d/SparseGaze/ADT-Gaze-structured
SparseGaze: /home/liumu/Github_Projects/SparseGaze/outputs/eval/adt/sparsegaze_cpf_forward_head_motion_residual_ss


## Configuration

In [3]:
REPORTS_DIR = eval_exp.DEFAULT_REPORTS_DIR
SPARSEGAZE_DIR = eval_exp.DEFAULT_SPARSEGAZE_DIR
HAGI_DIR = eval_exp.DEFAULT_HAGI_DIR
ADT_DATA_PATH = eval_exp.DEFAULT_ADT_DATA

adt_data = eval_exp.load_adt_data(ADT_DATA_PATH)
fps_options = eval_exp.sequence_names(
    adt_data=adt_data,
    fps=6,
    sparsegaze_dir=SPARSEGAZE_DIR,
)
print("default 6Hz sequence count:", len(fps_options))

default 6Hz sequence count: 10


## Method-Level Error Overview

This table evaluates one full sequence on missing/evaluated frames. It includes overall error and GT fixation/transition breakdown in a single table.

In [4]:
OVERVIEW_FPS = 6
OVERVIEW_SEQUENCE = eval_exp.sequence_names(
    adt_data=adt_data,
    fps=OVERVIEW_FPS,
    sparsegaze_dir=SPARSEGAZE_DIR,
)[0]
OVERVIEW_MODES = tuple(eval_exp.SPARSEGAZE_MODES.keys())

labels = eval_exp.load_event_labels(REPORTS_DIR, OVERVIEW_SEQUENCE)
start_frame, end_frame = eval_exp.resolve_window(labels, 0, None, 0)
predictions = eval_exp.load_method_event_errors(
    sequence=OVERVIEW_SEQUENCE,
    fps=OVERVIEW_FPS,
    modes=OVERVIEW_MODES,
    adt_data=adt_data,
    labels=labels,
    sparsegaze_dir=SPARSEGAZE_DIR,
    start_frame=start_frame,
    end_frame=end_frame,
    include_hagi=True,
    hagi_dir=HAGI_DIR,
)
summary = eval_exp.summarize_by_event(predictions)
method_summary = eval_exp.summarize_method_event_mae(predictions, summary)
print(f"sequence={OVERVIEW_SEQUENCE}")
print(f"fps={OVERVIEW_FPS}, frames={start_frame}..{end_frame}")
display(method_summary.round({
    "overall_mae_deg": 4,
    "overall_median_deg": 4,
    "overall_p90_deg": 4,
    "fixation_mae_deg": 4,
    "fixation_median_deg": 4,
    "fixation_p90_deg": 4,
    "transition_mae_deg": 4,
    "transition_median_deg": 4,
    "transition_p90_deg": 4,
    "transition_minus_fixation_mae_deg": 4,
}))

sequence=Apartment_release_decoration_skeleton_seq133_M1292
fps=6, frames=0..2736


,method,mode,overall_n,overall_mae_deg,overall_median_deg,overall_p90_deg,fixation_n,fixation_mae_deg,fixation_median_deg,fixation_p90_deg,transition_n,transition_mae_deg,transition_median_deg,transition_p90_deg,transition_minus_fixation_mae_deg
1,SparseGaze gt-repair,rollout_gt,2176,3.3834,1.1787,9.5163,789,1.0808,0.5797,2.1366,1387,4.6932,2.6216,12.1437,3.6125
2,SparseGaze linear,rollout_linear,2176,3.5131,1.2992,9.7447,789,1.1326,0.5994,2.2477,1387,4.8673,2.6674,12.8547,3.7346
3,SparseGaze pchip,rollout_pchip,2176,3.5247,1.3239,9.7702,789,1.1539,0.6071,2.2750,1387,4.8733,2.6396,12.7299,3.7194
4,SparseGaze rollout,rollout,2176,3.6233,1.4890,9.9239,789,1.3477,0.7345,3.0098,1387,4.9178,2.6921,12.5978,3.5701
0,HAGI++,hagi++,2069,3.6289,1.4022,10.2550,741,1.1794,0.6364,2.2668,1328,4.9958,2.7045,12.8555,3.8164


## Interactive Viewer

Select a sequence/window/method subset, then click `Render`. HAGI++ and SparseGaze variants are shown in one table and one figure. Error curves are drawn on each method's evaluated/missing frames and are broken at anchor gaps.


In [ ]:
def make_sparsegaze_event_widget():
    available_fps = [15, 10, 6]
    fps_dropdown = widgets.Dropdown(
        options=available_fps,
        value=6,
        description="FPS",
        layout=widgets.Layout(width="180px"),
    )
    seq_options = eval_exp.sequence_names(
        adt_data=adt_data,
        fps=fps_dropdown.value,
        sparsegaze_dir=SPARSEGAZE_DIR,
    )
    seq_dropdown = widgets.Dropdown(
        options=seq_options,
        value=seq_options[0],
        description="Sequence",
        layout=widgets.Layout(width="620px"),
    )
    labels_cache = {}
    prediction_cache = {}

    def labels_for(sequence):
        if sequence not in labels_cache:
            labels_cache[sequence] = eval_exp.load_event_labels(REPORTS_DIR, sequence)
        return labels_cache[sequence]

    def max_frame_for(sequence):
        return int(labels_for(sequence)["frame_index"].max())

    start_input = widgets.BoundedIntText(
        value=149,
        min=0,
        max=max_frame_for(seq_dropdown.value),
        step=1,
        description="Start",
        layout=widgets.Layout(width="180px"),
    )
    length_input = widgets.BoundedIntText(
        value=300,
        min=1,
        max=max_frame_for(seq_dropdown.value) + 1,
        step=1,
        description="Length",
        layout=widgets.Layout(width="180px"),
    )
    error_ymax_input = widgets.FloatText(
        value=0.0,
        description="Y max",
        layout=widgets.Layout(width="180px"),
    )
    mode_checkboxes = {
        mode: widgets.Checkbox(
            value=(mode in {"rollout", "rollout_pchip"}),
            description=label,
            indent=False,
            layout=widgets.Layout(width="220px"),
        )
        for mode, label in eval_exp.SPARSEGAZE_MODES.items()
    }
    hagi_checkbox = widgets.Checkbox(
        value=True,
        description="HAGI++",
        indent=False,
        layout=widgets.Layout(width="220px"),
    )
    render_button = widgets.Button(description="Render", button_style="primary")
    clear_cache_button = widgets.Button(description="Clear cache")
    status = widgets.HTML(value="")
    out = widgets.Output()

    def update_frame_bounds(sequence):
        max_frame = max_frame_for(sequence)
        start_input.max = max_frame
        length_input.max = max(max_frame + 1, 1)
        start_input.value = min(start_input.value, start_input.max)
        length_input.value = min(length_input.value, length_input.max)

    def on_fps_change(change):
        names = eval_exp.sequence_names(
            adt_data=adt_data,
            fps=change["new"],
            sparsegaze_dir=SPARSEGAZE_DIR,
        )
        seq_dropdown.options = names
        if names:
            seq_dropdown.value = names[0]
            update_frame_bounds(names[0])

    def on_seq_change(change):
        if change["new"]:
            update_frame_bounds(change["new"])

    def cached_predictions(sequence, fps, modes, labels, start_frame, end_frame, include_hagi):
        key = (sequence, int(fps), tuple(modes), int(start_frame), int(end_frame), bool(include_hagi))
        if key not in prediction_cache:
            prediction_cache[key] = eval_exp.load_method_event_errors(
                sequence=sequence,
                fps=fps,
                modes=modes,
                adt_data=adt_data,
                labels=labels,
                sparsegaze_dir=SPARSEGAZE_DIR,
                start_frame=start_frame,
                end_frame=end_frame,
                include_hagi=include_hagi,
                hagi_dir=HAGI_DIR,
            )
        return prediction_cache[key]

    def render(_=None):
        modes = tuple(mode for mode, checkbox in mode_checkboxes.items() if checkbox.value)
        with out:
            out.clear_output(wait=True)
            include_hagi = hagi_checkbox.value
            if not modes and not include_hagi:
                display("Select HAGI++ or at least one SparseGaze method.")
                return
            sequence = seq_dropdown.value
            fps = fps_dropdown.value
            labels = labels_for(sequence)
            start_frame, end_frame = eval_exp.resolve_window(
                labels,
                start_input.value,
                None,
                length_input.value,
            )
            status.value = f"Rendering {sequence}, {fps} Hz, frames {start_frame}..{end_frame}"
            predictions = cached_predictions(sequence, fps, modes, labels, start_frame, end_frame, include_hagi)
            if predictions.empty:
                display(f"No evaluated SparseGaze frames in {start_frame}..{end_frame}.")
                return
            summary = eval_exp.summarize_by_event(predictions)
            method_summary = eval_exp.summarize_method_event_mae(predictions, summary)
            display(method_summary.round({
                "overall_mae_deg": 4,
                "overall_median_deg": 4,
                "overall_p90_deg": 4,
                "fixation_mae_deg": 4,
                "fixation_median_deg": 4,
                "fixation_p90_deg": 4,
                "transition_mae_deg": 4,
                "transition_median_deg": 4,
                "transition_p90_deg": 4,
                "transition_minus_fixation_mae_deg": 4,
            }))
            fig = eval_exp.make_event_comparison_figure(
                sequence=sequence,
                fps=fps,
                labels=labels,
                predictions=predictions,
                summary=summary,
                start_frame=start_frame,
                end_frame=end_frame,
                error_ymax=error_ymax_input.value if error_ymax_input.value > 0 else None,
            )
            display(fig)
            plt.close(fig)
            status.value = f"Done. Cached windows: {len(prediction_cache)}"

    def clear_cache(_=None):
        prediction_cache.clear()
        status.value = "Cache cleared."

    fps_dropdown.observe(on_fps_change, names="value")
    seq_dropdown.observe(on_seq_change, names="value")
    render_button.on_click(render)
    clear_cache_button.on_click(clear_cache)

    ui = widgets.VBox([
        widgets.HBox([fps_dropdown, seq_dropdown]),
        widgets.HBox([start_input, length_input, error_ymax_input, render_button, clear_cache_button]),
        widgets.VBox([hagi_checkbox, *mode_checkboxes.values()], layout=widgets.Layout(border="1px solid #dddddd", padding="8px", width="260px")),
        status,
    ])
    display(ui, out)
    return ui, out

make_sparsegaze_event_widget();

Output()